# Load pretrained weights into your own model
Works for any model you've coded. Practice run: GPT-2 → your `GPT2Model`.

```
1 Download    weights (.safetensors) + config.json
2 Theirs      sizes + names + shapes
3 Yours       same sizes · names + shapes
4 Map + copy  match by name and shape · fix: copy · flip · split · reuse · skip
5 Test        output must match the original model
```
✍️ = your turn · solution at the bottom

## 0 · Setup
Installs `safetensors` (weight-file format) + `huggingface_hub` (downloader), and loads your model code.

In [1]:
%%capture
# %pip install safetensors huggingface_hub
%run GPT2Model.ipynb

## 1 · Download
`w` is a dict: name → numbers. Other model: change `REPO`. Works for split files too. Downloads once, then cached.
Models that need a login (e.g. Llama): run `hf auth login` once in a terminal.

In [2]:
from glob import glob
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

REPO = "openai-community/gpt2"
path = snapshot_download(REPO, allow_patterns=["model*.safetensors", "config.json"])
w = {}
for f in glob(f"{path}/model*.safetensors"):
    w.update(load_file(f))

/Users/faanjum/Developer/LLM-fine-tuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 415.44it/s]


In [3]:
# fetches from local huggingface cache - (~/.cache/huggingface/hub/models--openai-community--gpt2/snapshots/.../)
print(open(f"{path}/config.json").read())

{
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "resid_pdrop": 0.1,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "vocab_size": 50257
}


## 2 · GPT Actual Weights
Sizes (`config.json`), then names + shapes: block 0 + the rest (other blocks repeat).

In [4]:
import json, re

# Return True if the first number between dots is 0, or no such number exists
def block0(name):
    m = re.search(r"\.(\d+)\.", name)
    return m is None or m.group(1) == "0"

# d is a dictionary of weights, not a block number.
# Print only block 0 and unnumbered weights (like embeddings), sorted by name, with their shapes.
def show(d):
    for k, v in sorted(d.items()):
        if block0(k):
            print(f"{k:36} {list(v.shape)}")


# Print integer settings from config.json, such as layer count and hidden size.
print("config.json:\n",{k: v for k, v in json.load(open(f"{path}/config.json")).items() if isinstance(v, int)},"\n")  # sizes

print("Preview the weights in w:\n")
show(w)

config.json:
 {'bos_token_id': 50256, 'eos_token_id': 50256, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12, 'n_positions': 1024, 'summary_proj_to_labels': True, 'summary_use_proj': True, 'vocab_size': 50257} 

Preview the weights in w:

h.0.attn.bias                        [1, 1, 1024, 1024]
h.0.attn.c_attn.bias                 [2304]
h.0.attn.c_attn.weight               [768, 2304]
h.0.attn.c_proj.bias                 [768]
h.0.attn.c_proj.weight               [768, 768]
h.0.ln_1.bias                        [768]
h.0.ln_1.weight                      [768]
h.0.ln_2.bias                        [768]
h.0.ln_2.weight                      [768]
h.0.mlp.c_fc.bias                    [3072]
h.0.mlp.c_fc.weight                  [768, 3072]
h.0.mlp.c_proj.bias                  [768]
h.0.mlp.c_proj.weight                [3072, 768]
ln_f.bias                            [768]
ln_f.weight                          [768]
wpe.weight                           [1024, 768]
wte.weight         

## 3 · Our Model
Same sizes as theirs (step 2), e.g. GPT-2: `n_embd 768 · n_layer 12 · n_head 12`.
Theirs also has `c_attn.bias`, so `qkv_bias=True`.

In [5]:
model = GPT2Model({**model_config, "qkv_bias": True, "drop_rate": 0.0})
show(model.state_dict())

final_norm.scale                     [768]
final_norm.shift                     [768]
out_head.weight                      [50257, 768]
pos_emb.weight                       [1024, 768]
tok_emb.weight                       [50257, 768]
trf_blocks.0.attn.W_key.bias         [768]
trf_blocks.0.attn.W_key.weight       [768, 768]
trf_blocks.0.attn.W_query.bias       [768]
trf_blocks.0.attn.W_query.weight     [768, 768]
trf_blocks.0.attn.W_value.bias       [768]
trf_blocks.0.attn.W_value.weight     [768, 768]
trf_blocks.0.attn.mask               [1024, 1024]
trf_blocks.0.attn.out_proj.bias      [768]
trf_blocks.0.attn.out_proj.weight    [768, 768]
trf_blocks.0.ff.layers.0.bias        [3072]
trf_blocks.0.ff.layers.0.weight      [3072, 768]
trf_blocks.0.ff.layers.2.bias        [768]
trf_blocks.0.ff.layers.2.weight      [768, 3072]
trf_blocks.0.norm1.scale             [768]
trf_blocks.0.norm1.shift             [768]
trf_blocks.0.norm2.scale             [768]
trf_blocks.0.norm2.shift             

## 4 · Map + copy ✍️
Match each pair from steps 2–3 by similar words and shapes, then one `assign` per pair.

| Shapes | Fix |
|---|---|
| same | copy |
| reversed | flip: `.T` |
| theirs = 3 × yours | split: `.chunk(3, dim=-1)` |
| ours has no match  | reuse one (GPT-2: `out_head` = `wte`) |
| theirs has no match | skip |

In [6]:
import torch

done = set()
def assign(slot, numbers):  # copies, refuses wrong shapes, remembers what's filled
    assert slot.shape == numbers.shape, f"{list(slot.shape)} vs {list(numbers.shape)}"
    with torch.no_grad():
        slot.copy_(numbers)
    done.add(id(slot))

assign(model.tok_emb.weight, w["wte.weight"])                    # copy
for i, b in enumerate(model.trf_blocks):
    q, k, v = w[f"h.{i}.attn.c_attn.weight"].chunk(3, dim=-1)    # split
    assign(b.attn.W_query.weight, q.T)                           # flip
assign(model.out_head.weight, w["wte.weight"])                   # reuse

# ✍️ add the rest; step 5 lists what's missing

Not filled: 183 ['pos_emb.weight', 'trf_blocks.0.attn.W_query.bias', 'trf_blocks.0.attn.W_key.weight', 'trf_blocks.0.attn.W_key.bias', 'trf_blocks.0.attn.W_value.weight', 'trf_blocks.0.attn.W_value.bias', 'trf_blocks.0.attn.out_proj.weight', 'trf_blocks.0.attn.out_proj.bias', 'trf_blocks.0.ff.layers.0.weight', 'trf_blocks.0.ff.layers.0.bias', 'trf_blocks.0.ff.layers.2.weight', 'trf_blocks.0.ff.layers.2.bias', 'trf_blocks.0.norm1.shift', 'trf_blocks.0.norm1.scale', 'trf_blocks.0.norm2.shift', 'trf_blocks.0.norm2.scale', 'final_norm.shift', 'final_norm.scale']

In [ ]:
# Done:
# 1. Ours: attn.W_query.weight Theirs: h.0.attn.c_attn.weight

# Missing:
# ours: pos_emb.weight [1024, 768], Theirs: wpe.weight [1024, 768]



## 5 · Test
Lists what's not filled yet (block 0 shown), then generates. Correct weights give:
*"Every effort moves you forward. The first step is to understand the importance of your work"*

In [ ]:
import tiktoken

missing = [n for n, p in model.named_parameters() if id(p) not in done]
print(f"Not filled: {len(missing)}", [n for n in missing if block0(n)] if missing else "✅")
tok = tiktoken.get_encoding("gpt2")
ids = torch.tensor([tok.encode("Every effort moves you")])
model.eval()
with torch.no_grad():
    for _ in range(15):
        ids = torch.cat([ids, model(ids)[:, -1].argmax(dim=-1, keepdim=True)], dim=1)
print(tok.decode(ids[0].tolist()))

Not filled: 183 ['pos_emb.weight', 'trf_blocks.0.attn.W_query.bias', 'trf_blocks.0.attn.W_key.weight', 'trf_blocks.0.attn.W_key.bias', 'trf_blocks.0.attn.W_value.weight', 'trf_blocks.0.attn.W_value.bias', 'trf_blocks.0.attn.out_proj.weight', 'trf_blocks.0.attn.out_proj.bias', 'trf_blocks.0.ff.layers.0.weight', 'trf_blocks.0.ff.layers.0.bias', 'trf_blocks.0.ff.layers.2.weight', 'trf_blocks.0.ff.layers.2.bias', 'trf_blocks.0.norm1.shift', 'trf_blocks.0.norm1.scale', 'trf_blocks.0.norm2.shift', 'trf_blocks.0.norm2.scale', 'final_norm.shift', 'final_norm.scale']
Every effort moves you WARN WARN WARN WARNcatentrycatentrycatentrycatentryBioBiollahmiloactionDateactionDateactionDate


## Solution (try step 4 first)
Then run step 5 again.

In [ ]:
assign(model.tok_emb.weight, w["wte.weight"])
assign(model.pos_emb.weight, w["wpe.weight"])
for i, b in enumerate(model.trf_blocks):
    h = f"h.{i}."
    q, k, v = w[h + "attn.c_attn.weight"].chunk(3, dim=-1)
    assign(b.attn.W_query.weight, q.T); assign(b.attn.W_key.weight, k.T); assign(b.attn.W_value.weight, v.T)
    q, k, v = w[h + "attn.c_attn.bias"].chunk(3, dim=-1)
    assign(b.attn.W_query.bias, q); assign(b.attn.W_key.bias, k); assign(b.attn.W_value.bias, v)
    assign(b.attn.out_proj.weight, w[h + "attn.c_proj.weight"].T)
    assign(b.attn.out_proj.bias, w[h + "attn.c_proj.bias"])
    assign(b.ff.layers[0].weight, w[h + "mlp.c_fc.weight"].T)
    assign(b.ff.layers[0].bias, w[h + "mlp.c_fc.bias"])
    assign(b.ff.layers[2].weight, w[h + "mlp.c_proj.weight"].T)
    assign(b.ff.layers[2].bias, w[h + "mlp.c_proj.bias"])
    assign(b.norm1.scale, w[h + "ln_1.weight"]); assign(b.norm1.shift, w[h + "ln_1.bias"])
    assign(b.norm2.scale, w[h + "ln_2.weight"]); assign(b.norm2.shift, w[h + "ln_2.bias"])
assign(model.final_norm.scale, w["ln_f.weight"])
assign(model.final_norm.shift, w["ln_f.bias"])
assign(model.out_head.weight, w["wte.weight"])  # reuse: GPT-2 has no separate output layer